In [ ]:
import os
import random
import matplotlib.pyplot as plt
import torch
import torchvision

from pathlib import Path
from PIL import Image
from torch import nn
from torchvision.transforms import v2
from torchinfo import summary

from going_modular import engine

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42
DATA_DIR = Path('data/food-101-20-percent')
# GENERATOR = torch.Generator(DEVICE).manual_seed(RANDOM_SEED)

# torch.set_default_device(DEVICE)
if torch.cuda.get_device_capability() >= (8, 0):
    torch.backends.cuda.matmul.allow_tf32 = True


In [ ]:
DATA_DIR, DEVICE

In [ ]:
def create_vit(num_classes: int, compile: bool = False):
    """Creates a ViT-B/16 feature extractor model and transforms.

    Args:
        num_classes (int): number of target classes.
        seed (int): random seed value for output layer.
        compile (bool): whether to speed up PyTorch model.

    Returns:
        model (torch.nn.Module): ViT-B/16 feature extractor model. 
        model_transforms (torchvision.transforms): ViT-B/16 image transforms.
    """
    # Create ViT-B/16 pretrained weights, transforms, and model
    model_weights = torchvision.models.ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1
    model_transforms = model_weights.transforms()
    model = torchvision.models.vit_b_16(weights=model_weights)

    # Freeze all layers in model
    for param in model.parameters():
        param.requires_grad = False
    
    # Change classifier head to zero-initialized linear layer for fine-tuning
    model.heads = nn.Linear(in_features=768, out_features=num_classes)
    nn.init.zeros_(model.heads.weight)
    nn.init.zeros_(model.heads.bias)

    if compile:
        model = torch.compile(model)

    return model, model_transforms


In [ ]:
BRRRR = False

vit, vit_transforms = create_vit(num_classes=101, compile=BRRRR)

# summary(vit, 
#         input_size=(BATCH_SIZE, 3, 224, 224),
#         col_names=["input_size", "output_size", "num_params", "trainable"],
#         col_width=20,
#         row_settings=["var_names"])

In [ ]:
train_transforms = v2.Compose([
    v2.RandomHorizontalFlip(),
    v2.RandomRotation(degrees=(0, 180)),
    vit_transforms
])

train_transforms

In [ ]:
# train_set_full = torchvision.datasets.Food101(root=DATA_DIR,
#                                               split='train',
#                                               transform=train_transforms,
#                                               download=True)

# test_set = torchvision.datasets.Food101(root=DATA_DIR,
#                                         split='test',
#                                         transform=vit_transforms,
#                                         download=True)

train_set = torchvision.datasets.ImageFolder(root=DATA_DIR / 'train',
                                             transform=train_transforms)

val_set = torchvision.datasets.ImageFolder(root=DATA_DIR / 'validation',
                                           transform=vit_transforms)

test_set = torchvision.datasets.ImageFolder(root=DATA_DIR / 'test',
                                            transform=vit_transforms)

# train_set, val_set = torch.utils.data.random_split(train_set_full, 
#                                                    lengths=[0.98, 0.02],
#                                                    generator=torch.manual_seed(RANDOM_SEED))

len(train_set), len(val_set), len(test_set)

In [ ]:
CLASS_NAMES = train_set.classes

CLASS_NAMES[:10]
# len(CLASS_NAMES)

In [ ]:
BATCH_SIZE = 128

train_dataloader = torch.utils.data.DataLoader(dataset=train_set,
                                               batch_size=BATCH_SIZE,
                                               shuffle=True,
                                               pin_memory=True)

val_dataloader = torch.utils.data.DataLoader(dataset=val_set,
                                             batch_size=BATCH_SIZE,
                                             shuffle=False,
                                             pin_memory=True)

len(train_dataloader), len(val_dataloader)

In [ ]:
ETA = 0.01
MOMENTUM = 0.9
EPOCHS = 20

optimizer = torch.optim.SGD(params=vit.parameters(), lr=ETA, momentum=MOMENTUM)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer=optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

torch.manual_seed(RANDOM_SEED)
vit_results = engine.train(model=vit,
                           train_dataloader=train_dataloader,
                           test_dataloader=val_dataloader,
                           optimizer=optimizer,
                           scheduler=scheduler,
                           criterion=criterion,
                           epochs=EPOCHS,
                           device=DEVICE)

In [ ]:
epochs_range = range(len(vit_results['train_loss']))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

ax1.plot(epochs_range, vit_results['train_loss'], label='Train Loss')
ax1.plot(epochs_range, vit_results['test_loss'], label='Test Loss')
ax1.set_title('Loss')
ax1.set_xlabel('Epochs')
ax1.legend()

ax2.plot(epochs_range, vit_results['train_accuracy'], label='Train Accuracy')
ax2.plot(epochs_range, vit_results['test_accuracy'], label='Test Accuracy')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epochs')
ax2.legend()

plt.show()

In [ ]:
MODEL_DIR = Path('models')
MODEL_PATH = MODEL_DIR / 'pretrained_vit_feature_extractor_food101.pth'

torch.save(obj=vit.state_dict(), f=MODEL_PATH)

In [ ]:
def get_incorrect_preds(image_paths, transform):
    incorrect = []
    for image_path in image_paths:
        with Image.open(image_path) as img:
            with torch.inference_mode():
                if v2.ToTensor()(img).shape[0] == 1:
                    continue
                
                y_logits = vit(transform(img).unsqueeze(dim=0).to(DEVICE))
                pred_label = CLASS_NAMES[y_logits.argmax(dim=1).item()]
                target_label = image_path.parent.name

                if pred_label != target_label:
                    incorrect.append(image_path)
    
    return incorrect

In [ ]:
def plot_images(image_paths, transform):
    for image_path in image_paths:
        with Image.open(image_path) as img:
            fig, ax = plt.subplots()
            ax.imshow(img)
            ax.axis('off')

            with torch.inference_mode():
                y_logits = vit(transform(img).unsqueeze(dim=0).to(DEVICE))
                pred_probs = torch.softmax(y_logits, dim=1)
                pred_label = CLASS_NAMES[pred_probs.argmax(dim=1).item()]
                target_label = image_path.parent.name

                fig.suptitle(f'Label: {target_label} | Prediction: {pred_label}')
                ax.set_title(f'Prob: {round(pred_probs.max().item(), 3)}')

                plt.show()

In [ ]:
val_image_paths = get_incorrect_preds(list((DATA_DIR / 'validation').glob('*/*.jpg')), vit_transforms)

In [ ]:
# len(val_image_paths)
plot_images(val_image_paths[10:20], vit_transforms)

In [ ]:
test_dataloader = torch.utils.data.DataLoader(dataset=test_set,
                                             batch_size=BATCH_SIZE,
                                             shuffle=False,
                                             pin_memory=True)

test_loss, test_accuracy = engine.test_step(model=vit,
                                            dataloader=test_dataloader,
                                            criterion=nn.CrossEntropyLoss(),
                                            device=DEVICE)

test_loss, test_accuracy